# Anticipación del Conflicto Legal en Siniestros Laborales (NYSWCB)
### Notebook reproducible — Tesis de Maestría en Ciencia de Datos · ITBA 2026
**Autor:** Julián Tapia · **Director:** Ariel Aizemberg

Este notebook reproduce, de punta a punta y en pocos minutos, el pipeline central de la tesis:
**carga de datos → target de litigio evitable → validación temporal → CatBoost v3 → métricas (test 2022) → interacción AWW×C-3 → SHAP → impacto económico + Monte Carlo → auditoría de equidad (EEOC 4/5).**

> El benchmark completo de 9 modelos y la optimización con Optuna (60 trials, ~5,7 h en GPU) están en el repositorio (`src/`). Aquí se entrena directamente el modelo final con los hiperparámetros óptimos hallados, para que el notebook corra en una sesión estándar de Colab.

## 0 · Dependencias

In [ ]:
!pip -q install catboost shap scikit-learn pyarrow pandas numpy matplotlib > /dev/null
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
print("Listo")

## 1 · Cargar el dataset analítico

El dataset (`dataset_tesis_clean.parquet`, 1.592.919 reclamos × 25 columnas, ~25 MB) viene **incluido en el repositorio** y se descarga automáticamente. No hay que subir nada ni montar Drive.

> Es el subconjunto filtrado del [dataset público del NYSWCB](https://data.ny.gov/Government-Finance/Assembled-Workers-Compensation-Claims-Beginning-20/jshw-gkgu), guardado en Parquet (comprimido) para que entre en GitHub.

In [ ]:
import pandas as pd, urllib.request, os

URL   = "https://github.com/Jtapia2211/TesisITBA/raw/main/data/dataset_tesis_clean.parquet"
LOCAL = "data/dataset_tesis_clean.parquet"   # si se clonó el repo
path  = LOCAL if os.path.exists(LOCAL) else "dataset_tesis_clean.parquet"
if not os.path.exists(path):
    print("Descargando dataset desde el repositorio...")
    urllib.request.urlretrieve(URL, path)

df = pd.read_parquet(path)
print("Filas:", f"{len(df):,}", "| Columnas:", df.shape[1])
df.head(3)

## 2 · Target de litigio evitable y validación temporal (§3.3 · §4.15)

- **Target refinado = 1** si el caso judicializó **y** la resolución es no permanente (`NON-COMP`, `MED ONLY`, `TEMPORARY`). Los casos de incapacidad permanente / fallecimiento (litigio estructuralmente inevitable) pasan a 0.
- `claim_injury_type_REF` se usa **solo** para construir el target y luego se **excluye** de las features (anti-leakage, §3.4).
- **Split temporal estricto:** entrenamiento 2017–2020 · validación 2021 · prueba 2022.

In [ ]:
POS = {'2. NON-COMP','3. MED ONLY','4. TEMPORARY'}
df['target_ref'] = ((df['target']==1) & (df['claim_injury_type_REF'].isin(POS))).astype(int)

drop = ['target','target_ref','claim_injury_type_REF']
features = [c for c in df.columns if c not in drop]
cat_features = ['gender','accident_type','occupational_disease','county_of_injury','medical_fee_region',
                'wcio_cause_code','wcio_nature_code','wcio_body_code','carrier_type','district_name',
                'industry_code','industry_desc']
cat_features = [c for c in cat_features if c in features]
num_features = [c for c in features if c not in cat_features]
print(f"{len(features)} features ({len(cat_features)} categóricas, {len(num_features)} numéricas)")

tr = df[df.accident_year<=2020].copy()
va = df[df.accident_year==2021].copy()
te = df[df.accident_year==2022].copy()
for name,d in [('train',tr),('val',va),('test',te)]:
    print(f"{name:5s} n={len(d):>9,}  prevalencia target_ref={d.target_ref.mean()*100:5.2f}%")

In [ ]:
# Preprocesamiento: categóricas->str, numéricas->mediana del train
def prep(d):
    d = d.copy()
    for c in cat_features: d[c] = d[c].astype(str).fillna('NA')
    for c in num_features: d[c] = pd.to_numeric(d[c], errors='coerce')
    return d
tr,va,te = prep(tr),prep(va),prep(te)
med = tr[num_features].median()
for d in (tr,va,te): d[num_features] = d[num_features].fillna(med)

Xtr,ytr = tr[features], tr['target_ref']
Xva,yva = va[features], va['target_ref']
Xte,yte = te[features], te['target_ref']
spw = (ytr==0).sum()/(ytr==1).sum()
print("scale_pos_weight (train) =", round(spw,3))

## 3 · Modelo final CatBoost v3 (§4.19.2 · §4.25)

Hiperparámetros óptimos hallados por la búsqueda bayesiana (Optuna, §4.20–4.27):
`depth=10 · learning_rate=0.061 · l2_leaf_reg=7.62 · bagging_temperature=0.79 · border_count=82`.

In [ ]:
from catboost import CatBoostClassifier, Pool
try:
    from catboost.utils import get_gpu_device_count
    TASK = 'GPU' if get_gpu_device_count() > 0 else 'CPU'
except Exception:
    TASK = 'CPU'
print("Entrenando en:", TASK)

pool_tr = Pool(Xtr, ytr, cat_features=cat_features)
pool_va = Pool(Xva, yva, cat_features=cat_features)
pool_te = Pool(Xte, yte, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=1500, depth=10, learning_rate=0.061, l2_leaf_reg=7.62,
    bagging_temperature=0.79, border_count=82, loss_function='Logloss',
    eval_metric='PRAUC', scale_pos_weight=spw, random_seed=42,
    task_type=TASK, early_stopping_rounds=100, verbose=200)
model.fit(pool_tr, eval_set=pool_va)
print("Iteraciones efectivas:", model.tree_count_)

## 4 · Evaluación en prueba 2022 (§4.19.2 · Abstract)

Valores esperados: **AUC-ROC ≈ 0,883 · PR-AUC ≈ 0,598 · Recall ≈ 67,8 % · Precisión ≈ 51,9 %** a τ=0,708.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
p_te = model.predict_proba(Xte)[:,1]
auc = roc_auc_score(yte, p_te); prauc = average_precision_score(yte, p_te)
# KS
from scipy.stats import ks_2samp
ks = ks_2samp(p_te[yte==1], p_te[yte==0]).statistic
TAU = 0.708
pred = (p_te>=TAU).astype(int)
tn,fp,fn,tp = confusion_matrix(yte,pred).ravel()
recall = tp/(tp+fn); prec = tp/(tp+fp)
print(f"AUC-ROC : {auc:.4f}")
print(f"PR-AUC  : {prauc:.4f}")
print(f"KS      : {ks:.4f}")
print(f"\nMatriz @ τ={TAU}:  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"Recall  : {recall*100:.1f}%   Precisión: {prec*100:.1f}%")

## 5 · Interacción Salario × Formulario C-3 (§4.33 · Tabla 31)

Tasa de judicialización real (prueba 2022) por las cuatro combinaciones AWW(=0/>0) × C-3.
Valores esperados: **3,2 % / 26,7 % / 26,2 % / 49,0 %**.

In [ ]:
te2 = te.copy(); te2['AWW>0'] = (pd.to_numeric(te2['aww'],errors='coerce')>0).astype(int)
seg = (te2.groupby(['AWW>0','has_C3'])
          .agg(n=('target_ref','size'), tasa=('target_ref','mean')))
seg['tasa %'] = (seg['tasa']*100).round(1); seg['share %']=(seg['n']/len(te2)*100).round(1)
print(seg[['n','tasa %','share %']])
hi = te2[(te2['AWW>0']==1)&(te2.has_C3==1)].target_ref.sum()
print(f"\nAWW>0 & C-3 concentra el {hi/te2.target_ref.sum()*100:.0f}% de los litigios reales")

## 6 · Interpretabilidad SHAP (§4.28–4.35)

In [ ]:
import shap, numpy as np
# Muestra estratificada 50/50 como en la tesis (§4.28): 4.000 positivos + 4.000 negativos
pos = te[te.target_ref==1].sample(min(4000,(te.target_ref==1).sum()), random_state=0)
neg = te[te.target_ref==0].sample(4000, random_state=0)
samp = pd.concat([pos, neg])

expl = model.get_feature_importance(
    Pool(samp[features], samp['target_ref'], cat_features=cat_features), type='ShapValues')
sv = expl[:, :-1]
order = np.argsort(np.abs(sv).mean(0))[::-1][:10]
for i in order:
    print(f"{features[i]:24s}  mean|SHAP| = {np.abs(sv[:,i]).mean():.4f}")

## 7 · Impacto económico (§5.2 · §5.5)

Parámetros: `C_j=$13.000` (litigio evitable) · `C_p=$3.500` (intervención) · `C_fp=$500` (falsa alarma).
Cota superior (efectividad plena) ≈ **\$252 M, ROI 2,28×**. Estimación central Monte Carlo ≈ **\$100 M, ROI 0,9×**.

In [ ]:
Cj,Cp,Cfp = 13000,3500,500
ahorro = tp*(Cj-Cp) - fp*Cfp
inv = tp*Cp + fp*Cfp
print(f"Cota superior (p_éxito=100%): ahorro ${ahorro/1e6:.1f}M | inversión ${inv/1e6:.1f}M | ROI {ahorro/inv:.2f}x")
print(f"Reducción vs sin modelo: {ahorro/((tp+fn)*Cj)*100:.1f}%")

# Monte Carlo: incertidumbre en costos y efectividad de la mediación
rng = np.random.default_rng(42); N=50000
pj  = rng.triangular(10000,13000,18000,N)
pp  = rng.triangular(2500,3500,4500,N)
pfp = rng.triangular(300,500,800,N)
pex = rng.triangular(0.40,0.60,0.75,N)
mc = tp*(pex*pj - pp) - fp*pfp
print(f"\nMonte Carlo: P50 ${np.percentile(mc,50)/1e6:.0f}M  [P5 ${np.percentile(mc,5)/1e6:.0f}M – P95 ${np.percentile(mc,95)/1e6:.0f}M]")
print(f"P(ahorro>0) = {(mc>0).mean()*100:.1f}%   ROI mediano = {np.percentile(mc,50)/inv:.2f}x")

## 8 · Auditoría de equidad — regla EEOC 4/5 (§6.3.1)

Equal Opportunity (igualdad de TPR) por género y por quintil de salario (AWW). La disparidad es significativa si el ratio entre grupos < 0,80.

In [ ]:
te3 = te.copy(); te3['pred']=pred; te3['y']=yte.values
def tpr(d): 
    pos=d[d.y==1]; return pos.pred.mean() if len(pos) else np.nan
print("== Por género ==")
g = te3.groupby('gender').apply(lambda d: pd.Series({'n':len(d),'TPR':tpr(d)}))
g=g[g.n>500]; print(g.round(3))
print(f"Ratio TPR(F)/TPR(M) = {g.loc['F','TPR']/g.loc['M','TPR']:.3f}")

print("\n== Por quintil de AWW (sobre AWW>0) ==")
pos = te3[pd.to_numeric(te3.aww,errors='coerce')>0].copy()
pos['Q'] = pd.qcut(pd.to_numeric(pos.aww,errors='coerce'),5,labels=[1,2,3,4,5])
q = pos.groupby('Q').apply(lambda d: pd.Series({'n':len(d),'TPR':tpr(d)}))
q['ratio_vs_Q1'] = (q.TPR/q.TPR.iloc[0]).round(3)
print(q.round(3))
print("\nQ5/Q1 =", round(q.TPR.iloc[-1]/q.TPR.iloc[0],3),
      "→", "✔ cumple EEOC" if q.TPR.iloc[-1]/q.TPR.iloc[0]>=0.8 else "⚠ viola EEOC (ver calibración per-quintil §6.3.1)")

---
### Notas
- Las cifras económicas son proyecciones bajo supuestos explícitos, no validadas en producción.
- La calibración diferenciada del umbral por quintil (`fairness_calibration.py`, §6.3.1) restablece el cumplimiento EEOC en Q5; aquí se muestra el estado pre-calibración.
- Pipeline completo, benchmark de 9 modelos y Optuna: ver `src/` en el repositorio.